# Practical 1 -- Tensor Basics

In [1]:
import ipywidgets as widgets
from IPython.display import display, HTML

import torch
import torch.nn as nn

Tensors are the basic building block for storing everything: parameters, gradients, optimizer states, data, activations.

We can create tensors like this:

In [2]:
x = torch.tensor([[1., 2, 3], [4, 5, 6]])  # matrix with spec. values
x = torch.zeros(4, 8)  # 4x8 matrix of all zeros
x = torch.ones(4, 8)  # 4x8 matrix of all ones
x = torch.randn(4, 8)  # 4x8 matrix of iid Normal(0, 1) samples

# Or allocate uninitialized memory:
x = torch.empty(4, 8)  # 4x8 matrix of uninitialized memory
# and initilize later: i.e.: fill x with truncated normal samples
nn.init.trunc_normal_(x, mean = 2, std=3, a=-2, b=2)
print(x)

tensor([[-0.4824, -1.6033,  0.4650, -0.5283, -1.0779, -0.3618, -0.1887,  1.9753],
        [ 0.5759, -0.6407, -0.6776,  0.4498, -0.5191,  1.8634,  1.0081,  0.8218],
        [ 1.9809,  1.6358,  1.5302, -0.0482,  1.3755,  0.2596, -0.9678, -0.4142],
        [ 0.9692,  1.2656,  1.0027,  0.4495, -0.7142, -1.8644, -0.1106, -0.0739]])


We can put them on a GPU

In [3]:
## Add demonstration for GPU loading

## A) Precision
Almost all tensors we work with (model parameters, gradient, activations, optimimizer states) are stored as floating point numbers.

#### Single precision (float32)
The default type is ``torch.float32'', i.e., 32-bits = 4 bytes per number.


<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/d/d2/Float_example.svg/960px-Float_example.svg.png" alt="Floating Point Example" width="600"/>

In [4]:
print(f'Variable x.dtype = {x.dtype}')
print(f'Variable x has {x.numel()} elements and each element takes {x.element_size()} bytes')

def get_memory_usage(x: torch.Tensor):
    return x.numel() * x.element_size()

print(f'Variable x uses {get_memory_usage(x)} bytes.')

# Let's check that the memory usage is what we expect for a 12288 x 12288 matrix of 32-bit floats (4 bytes per element):
print(f'Memory usage of a 12288 x 12288 matrix of 32-bit floats: {get_memory_usage(torch.empty(12288 * 4, 12288)) / (1024 * 1024 * 1024)} GB.')

Variable x.dtype = torch.float32
Variable x has 32 elements and each element takes 4 bytes
Variable x uses 128 bytes.
Memory usage of a 12288 x 12288 matrix of 32-bit floats: 2.25 GB.


#### Half-precision (float16)
The float16 data type (half-precision) cuts down the memory:

<img src="https://cs336.stanford.edu/spring2025-lectures/images/fp16.png" alt="Floating Point Example" width="600"/>

In [5]:
assert(get_memory_usage(torch.empty(12288 * 4, 12288)) == 2304 * 1024 * 1024)

#### Brain floating point (bfloat16)
The bfloat data type (half-precision) cuts down the memory:

<img src="https://cs336.stanford.edu/spring2025-lectures/images/bf16.png" alt="Floating Point Example" width="600"/>

#### Low-precision floating point (fp8)
New FP8 standard developed by NVIDIA motivated by ML workloads. See [NVIDIA FP8 Primer](https://docs.nvidia.com/deeplearning/transformer-engine/user-guide/examples/fp8_primer.html) for details.

<img src="https://docs.nvidia.com/deeplearning/transformer-engine/user-guide/_images/fp8_formats.png" alt="Floating Point Example" width="600"/>

- H100 supports two FP8 formats:
  - **E4M3** (1 sign, 4 exponent, 3 mantissa): higher precision, smaller range (≈ **±448**, plus NaN).
  - **E5M2** (1 sign, 5 exponent, 2 mantissa): larger range (≈ **±57344**, plus **±inf** and NaN).


In [6]:
dtype_e4m3 = torch.float8_e4m3fn
info_e4m3 = torch.finfo(dtype_e4m3)

# 2. E5M2 (5 exponent bits, 2 mantissa bits)
dtype_e5m2 = torch.float8_e5m2
info_e5m2 = torch.finfo(dtype_e5m2)

print("--- Verifying FP8 Ranges ---")
print(f"E4M3 Max/Min: [{info_e4m3.min}, {info_e4m3.max}]")
print(f"E5M2 Max/Min: [{info_e5m2.min}, {info_e5m2.max}]")

# --- Casting Example ---
# Create a standard 32-bit floating point tensor
fp32_tensor = torch.tensor([1.5, 42.0, -300.0, 500.0], dtype=torch.float32)

# Cast the tensor to the two different FP8 formats
fp8_e4m3_tensor = fp32_tensor.to(dtype_e4m3)
fp8_e5m2_tensor = fp32_tensor.to(dtype_e5m2)

print("\n--- Tensor Casting ---")
print(f"Original FP32: {fp32_tensor}")
print(f"Cast to E4M3:  {fp8_e4m3_tensor}")
print(f"Cast to E5M2:  {fp8_e5m2_tensor}")

--- Verifying FP8 Ranges ---
E4M3 Max/Min: [-448.0, 448.0]
E5M2 Max/Min: [-57344.0, 57344.0]

--- Tensor Casting ---
Original FP32: tensor([   1.5000,   42.0000, -300.0000,  500.0000])
Cast to E4M3:  tensor([   1.5000,   40.0000, -288.0000,       nan], dtype=torch.float8_e4m3fn)
Cast to E5M2:  tensor([   1.5000,   40.0000, -320.0000,  512.0000], dtype=torch.float8_e5m2)


#### Training: Precision tradeoff
- **Higher precision (float32):** more accurate/stable, **more memory**, **more compute**
- **Lower precision (bf16 / fp8):** faster and smaller, but **less accurate/stable**

**Idea:** keep computations *as low-precision as safely possible*, but retain **float32** where it matters for stability.
- Use **{bf16, fp8}** for the **forward pass**
- Keep **float32** “master” state where stability matters most:
  - optimizer state (e.g. Adam moments)
  - weight updates (often via FP32 master weights)
  - some reductions / norms (depending on implementation)
- Gradients are often accumulated in higher precision; the guiding principle is: **accumulate/scale in FP32 when needed to avoid under/overflow.**

## B) Tensors Storage
PyTorch Tensors are pointers to allocated memory

... + metadata describing how to get any elements of the tensor.

<img src="https://cs336.stanford.edu/spring2025-lectures/var/files/image-97aa05a6701b46521cb8a7c1e096c7e7-https_martinlwx_github_io_img_2D_tensor_strides_png" alt="Floating Point Example" width="600"/>

In [7]:
x = torch.tensor([
        [0., 1, 2, 3],
        [4, 5, 6, 7],
        [8, 9, 10, 11],
        [12, 13, 14, 15],
    ])

# To go to the next row (dim 0), skip 4 elements in storage.
print(f'Stride of dim 0: {x.stride(0)}, of dim 1: {x.stride(1)}')

# To get to an element (1,2):
r, c = 1, 2
index = r * x.stride(0) + c * x.stride(1)
print(f'Index in storage for element (1,2): {index}, value: {x.reshape(-1)[index].item()}')

Stride of dim 0: 4, of dim 1: 1
Index in storage for element (1,2): 6, value: 6.0


In [8]:
# 1. Math Display Text
math_display = widgets.HTML(value="<h3 style='color: #333;'>Click a cell in the 2D grid!</h3>")

# 2. Build Physical Storage (1D Array)
cells_1d = [widgets.Button(description=str(i), layout=widgets.Layout(width='45px', height='45px')) for i in range(16)]
for btn in cells_1d:
    btn.style.button_color = '#e8f5e9' # Light green color

box_1d = widgets.HBox(cells_1d)

# 3. Build Logical View (2D Array) and Click Logic
cells_2d = []

def create_callback(index):
    def on_click(b):
        r = index // 4
        c = index % 4
        
        # --- NEW HIGHLIGHTED STRIDE TEXT ---
        stride_style = "background-color: #fff59d; color: #000; padding: 2px 6px; border-radius: 4px; border: 1px solid #fbc02d;"
        math_display.value = (
            f"<h3 style='color: #d32f2f;'>"
            f"Row [{r}] × <span style='{stride_style}'>Stride(4)</span> + "
            f"Col [{c}] × <span style='{stride_style}'>Stride(1)</span> "
            f"= Storage Index [{index}]"
            f"</h3>"
        )
        # -----------------------------------
        
        # Reset all colors
        for i in range(16):
            cells_2d[i].style.button_color = '#e3f2fd'
            cells_1d[i].style.button_color = '#e8f5e9'
        
        # Highlight the clicked cells
        cells_2d[index].style.button_color = '#1976d2'
        cells_1d[index].style.button_color = '#388e3c'
        
    return on_click

for i in range(16):
    btn = widgets.Button(description=str(i), layout=widgets.Layout(width='60px', height='60px'))
    btn.style.button_color = '#e3f2fd' # Light blue color
    btn.on_click(create_callback(i))
    cells_2d.append(btn)

# Arrange the 2D cells into a 4x4 Grid
grid_2d = widgets.GridBox(cells_2d, layout=widgets.Layout(grid_template_columns="repeat(4, 65px)"))

# 4. Display Everything in the Notebook
display(HTML("<h3 style='color: #333;'>Logical View (4x4)</h3>"))
display(grid_2d)
display(math_display)
display(HTML("<h3 style='color: #333;'>Physical Storage (1x16)</h3>"))
display(box_1d)

GridBox(children=(Button(description='0', layout=Layout(height='60px', width='60px'), style=ButtonStyle(button…

HTML(value="<h3 style='color: #333;'>Click a cell in the 2D grid!</h3>")

### Tensors Slicing & Views

Many ops return a **view**: a tensor that **shares storage** with the original (O(1), no copy).
- Views: slicing/indexing, `transpose`, `view` (if stride-compatible), sometimes `reshape`.

**Why you care (LLMs):**
Transformers constantly reshape/transpose tensors (e.g. `[B,T,C] → [B,H,T,D]`).  
Accidentally forcing copies (e.g. via `.contiguous()`) can add big memory + time overhead.

**Contiguity & strides**
- `transpose` often creates a **non-contiguous** view (different strides).
- `.view(...)` requires a contiguous/stride-compatible layout → may error on non-contiguous tensors.
- `.reshape(...)` is safer: it returns a view *if possible*, otherwise it makes a copy.

**Rule of thumb:** prefer views; call `.contiguous()` only when an op truly requires it.

**The Math of Contiguity:** A tensor is truly contiguous only if its memory layout has no "gaps."
- **The Rule:** The last stride = `1`, other strides must be the size of the next dimension multiplied by the next stride: `Stride[i] = Shape[i+1] * Stride[i+1]`.
- **Example:** A tensor of shape `[Batch=2, Time=3, Channels=4]` is contiguous if its strides are exactly `[12, 4, 1]`.

In [9]:
def same_storage(a, b):
    return a.untyped_storage().data_ptr() == b.untyped_storage().data_ptr()

x = torch.tensor([[1., 2, 3], [4, 5, 6]])

# Views share storage (no copy)
t = x.transpose(1, 0)          # non-contiguous view
print("shares storage:", same_storage(x, t))
print("contiguous?", t.is_contiguous(), "stride:", t.stride())

# Mutations propagate through views
x[0, 0] = 100.
print("t[0,0] after mutating x:", t[0, 0].item())

# view() can fail on non-contiguous tensors; reshape() is safer
try:
    t.view(2, 3)
except RuntimeError as e:
    print("view failed:", str(e).split("\n")[0])

r = t.reshape(2, 3)            # view if possible else copy
y = t.contiguous().view(2, 3)  # forces a copy, then view works
print("reshape shares storage?", same_storage(x, r))
print("contiguous().view shares storage?", same_storage(x, y))

shares storage: True
contiguous? False stride: (1, 3)
t[0,0] after mutating x: 100.0
view failed: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.
reshape shares storage? False
contiguous().view shares storage? False


In [10]:
import ipywidgets as widgets
from IPython.display import display, HTML

# --- UI Setup ---
status_html = widgets.HTML(value="")

# Create 8 Logical Buttons and 8 Physical Buttons
logical_btns = [widgets.Button(description=str(i), layout=widgets.Layout(width='60px', height='60px')) for i in range(8)]
physical_btns = [widgets.Button(description=str(i), layout=widgets.Layout(width='45px', height='45px')) for i in range(8)]

logical_grid = widgets.GridBox(logical_btns, layout=widgets.Layout(grid_template_columns="repeat(4, 65px)"))
physical_box = widgets.HBox(physical_btns)

# --- State Data ---
states = {
    'original': {
        'cols': 4,
        'logical': [0, 1, 2, 3, 4, 5, 6, 7],
        'physical': [0, 1, 2, 3, 4, 5, 6, 7],
        'status': "<h4 style='color: #2e7d32;'>Original [Contiguous] - Strides: (4, 1)</h4>"
    },
    'transposed': {
        'cols': 2,
        'logical': [0, 4, 1, 5, 2, 6, 3, 7],
        'physical': [0, 1, 2, 3, 4, 5, 6, 7], # Storage is UNTOUCHED
        'status': "<h4 style='color: #1565c0;'>Transposed View [Non-Contiguous] - Storage Shared! Strides: (1, 4)</h4>"
    },
    'contiguous': {
        'cols': 2,
        'logical': [0, 4, 1, 5, 2, 6, 3, 7],
        'physical': [0, 4, 1, 5, 2, 6, 3, 7], # Storage is REALLOCATED
        'status': "<h4 style='color: #c62828;'>Made Contiguous [Copied] - Memory Reallocated! Strides: (2, 1)</h4>"
    }
}

current_state = 'original'

def reset_colors():
    for btn in logical_btns:
        btn.style.button_color = '#e3f2fd' # Light blue
        btn.style.text_color = 'black'
    for btn in physical_btns:
        btn.style.button_color = '#e8f5e9' # Light green
        btn.style.text_color = 'black'

def set_state(state_name):
    global current_state
    current_state = state_name
    state = states[state_name]
    
    # Update Grid layout dimensions (2x4 vs 4x2)
    logical_grid.layout.grid_template_columns = f"repeat({state['cols']}, 65px)"
    
    # Update Number Labels
    for i in range(8):
        logical_btns[i].description = str(state['logical'][i])
        physical_btns[i].description = str(state['physical'][i])
    
    status_html.value = state['status']
    reset_colors()

# --- Control Buttons ---
btn_orig = widgets.Button(description="1. Original (2x4)", button_style='success')
btn_trans = widgets.Button(description="2. Transpose (View)", button_style='info')
btn_contig = widgets.Button(description="3. Contiguous (Copy)", button_style='danger')

btn_orig.on_click(lambda b: set_state('original'))
btn_trans.on_click(lambda b: set_state('transposed'))
btn_contig.on_click(lambda b: set_state('contiguous'))

controls = widgets.HBox([btn_orig, btn_trans, btn_contig])

# --- Click Interaction for Memory Mapping ---
def make_click_handler(idx):
    def on_click(b):
        reset_colors()
        
        # 1. Highlight the clicked logical cell
        logical_btns[idx].style.button_color = '#1976d2'
        logical_btns[idx].style.text_color = 'white'
        
        # 2. Find the actual numerical value of that cell
        val = states[current_state]['logical'][idx]
        
        # 3. Find where that value currently lives in physical RAM
        phys_idx = states[current_state]['physical'].index(val)
        
        # 4. Highlight the physical RAM cell
        physical_btns[phys_idx].style.button_color = '#388e3c'
        physical_btns[phys_idx].style.text_color = 'white'
        
    return on_click

for i in range(8):
    logical_btns[i].on_click(make_click_handler(i))

# --- Initial Render ---
set_state('original')

# --- Display Everything in Notebook ---
display(HTML("<h3 style='color: #333;'>1. Logical View (Tensor Shape)</h3>"))
display(HTML("<p style='color: #666;'><i>Click a cell below to trace its pointer to physical memory!</i></p>"))
display(logical_grid)
display(controls)
display(status_html)
display(HTML("<h3 style='color: #333; margin-top: 20px;'>2. Physical Storage (RAM)</h3>"))
display(physical_box)

GridBox(children=(Button(description='0', layout=Layout(height='60px', width='60px'), style=ButtonStyle(button…

HTML(value="<h4 style='color: #2e7d32;'>Original [Contiguous] - Strides: (4, 1)</h4>")

## C) Tensor operations